# D1~D2 PoC 노트북

본격 모듈화 전, 각 외부 의존성이 동작하는지 빠르게 확인하는 노트북.

## 검증 항목
1. OpenAI API JSON 모드로 한국어 분류 → 성공
2. yt-dlp 메타데이터 추출 → 성공
3. OpenGraph 파서로 네이버 블로그 → 성공
4. pgvector 임베딩 INSERT·SELECT → 성공
5. Supabase 클라이언트 → 성공

여기서 검증된 코드는 `app/services/*.py` 로 옮긴다.

In [ ]:
# 환경 세팅
import os
from dotenv import load_dotenv
load_dotenv()

print('OpenAI key set:', bool(os.getenv('OPENAI_API_KEY')))
print('Supabase url set:', bool(os.getenv('SUPABASE_URL')))

## 1. OpenAI JSON 모드 — 한국어 분류

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

test_meta = {
    'title': '10분 만에 끝나는 초보용 홈트',
    'description': '운동 초보를 위한 짧고 강력한 홈트 루틴',
    'platform': 'youtube',
}

resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': '카테고리·태그·요약을 JSON으로만 응답하세요. 카테고리는 FOOD/TRAVEL/FITNESS/...'},
        {'role': 'user', 'content': str(test_meta)},
    ],
    response_format={'type': 'json_object'},
    temperature=0.3,
)
result = json.loads(resp.choices[0].message.content)
print(result)

## 2. yt-dlp 메타 추출

In [ ]:
from yt_dlp import YoutubeDL

with YoutubeDL({'quiet': True, 'skip_download': True}) as ydl:
    info = ydl.extract_info('https://www.youtube.com/watch?v=dQw4w9WgXcQ', download=False)

print('Title:', info.get('title'))
print('Channel:', info.get('uploader'))
print('Thumbnail:', info.get('thumbnail'))

## 3. OpenGraph 파서

In [ ]:
import httpx
from bs4 import BeautifulSoup

r = httpx.get('https://www.naver.com/', follow_redirects=True, timeout=10,
              headers={'User-Agent': 'Mozilla/5.0'})
soup = BeautifulSoup(r.text, 'html.parser')

og_title = soup.find('meta', {'property': 'og:title'})
og_desc = soup.find('meta', {'property': 'og:description'})
print('OG title:', og_title.get('content') if og_title else None)
print('OG desc:', og_desc.get('content') if og_desc else None)

## 4. 임베딩 + 코사인 유사도 비교

In [ ]:
import numpy as np

def embed(text):
    return client.embeddings.create(model='text-embedding-3-small', input=text).data[0].embedding

a = np.array(embed('초보자용 홈트 영상'))
b = np.array(embed('운동 초보를 위한 10분 루틴'))
c = np.array(embed('일본 교토 카페 추천'))

def cos(x, y):
    return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))

print('운동 ↔ 운동 유사도:', cos(a, b))
print('운동 ↔ 카페 유사도:', cos(a, c))